# Run the experiment on Google Colab

Open this notebook from its GitHub link in every session, on a GPU runtime (Runtime > Change
runtime type), so the cells are always those on `main`:
<https://colab.research.google.com/github/meiazero/garimpo-uncertainty/blob/main/notebooks/01_run_experiment.ipynb>

1. Run Runtime > Run all and keep the tab open.
2. Read the verdicts at the end of section 5.

There is nothing to set. Each session runs the tests once per commit, then the pilot until all four
models pass on the commit checked out, then the four full runs, which resume where the last session
stopped. Only after a code fix to a stage the full runs already finished, set the one option of
section 1. Every cell can run again: Run all a second time, in the same session or a new one, skips
what is done and resumes the rest. The last cell saves everything to Drive and releases the GPU, so the
session stops spending compute units. The data must already be in Drive as
`MyDrive/garimpo/data.tar.zst`; the notebook downloads nothing and analyzes nothing.

| Section | What it does |
|---|---|
| 1. Redo after a fix | The one option: a stage the full runs redo after a code fix |
| 2. Setup | Clones the code, builds the environment |
| 3. Drive and data | Mounts Drive, unpacks the data once, restores earlier runs and logs |
| 4. Tests | The test suite on the GPU, once per commit |
| 5. Runs | `garimpo experiment`: the pilot, then the full runs, a verdict per model, the merge |
| 6. Save | The last copy to Drive |
| 7. Release | The GPU released |

## 1. Redo after a fix

Leave `FORCE_FROM` empty. A run skips what it finished and resumes the rest, and after a code
change the pilot redoes itself from training on its own, since it only checks the code.

Set it only after a code fix to a stage the full runs already finished: choose that stage, and the
four full runs redo it and every stage after it. It refuses `train` while `last.pt` holds a training
that has not finished. The value lasts while the tab is open; reopening the notebook from its link
empties it. Every other value comes from `configs/`, versioned with the code: change it on your own
machine, push, and the next session clones it.

In [ ]:
# @title The one option
FORCE_FROM = ""  # @param ["", "train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics", "search_profile"]

BRANCH = "main"  # the experiment runs the code on main
RELEASE = True  # the last cell always releases the GPU
OPTIONS = f"--force-from {FORCE_FROM}" if FORCE_FROM else ""
print(f"The full runs redo {'from ' + FORCE_FROM if FORCE_FROM else 'nothing'}.")

## 2. Setup

The runtime's disk is erased when a session ends, so this cell clones the code again, or moves the
clone already there to the last commit of `BRANCH`, and installs `requirements.txt`. The install
stops, and names the package, when pip would replace Colab's torch or a CUDA library, which pick
the GPU's kernels: choose the latest runtime version then. Every later cell calls a `garimpo`
command, which loads the code just installed. When a step fails, `SETUP_OK` is false and
the cells that run the code skip, so none runs an older install; the release still runs.

In [ ]:
# @title Setup: clone BRANCH and install it
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/meiazero/garimpo-uncertainty.git"
CODE = "/content/garimpo-uncertainty"  # the clone; erased with the session
steps = [] if Path(CODE, ".git").is_dir() else [["git", "clone", "-q", REPO, CODE]]
steps += [
    ["git", "-C", CODE, "fetch", "-q", "origin", BRANCH],
    ["git", "-C", CODE, "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"],
    ["git", "-C", CODE, "submodule", "update", "-q", "--init", "--recursive"],
    [sys.executable, f"{CODE}/scripts/colab_install.py"],
    ["git", "-C", CODE, "log", "-1", "--format=Running commit %h %cs %s"],
]
# all() stops at the first failure; a later cell then skips rather than run an older install.
SETUP_OK = all(subprocess.run(step, check=False).returncode == 0 for step in steps)  # noqa: S603
if Path(CODE).is_dir():
    os.chdir(CODE)
if not SETUP_OK:
    print("The setup failed, so the cells that run the code skip: read the error above.")

## 3. Drive and data

The store is unpacked from Drive only when `DATA` does not hold a complete unpack yet, marked by
`.unpacked` once tar succeeds: it takes 69.2 GB, for all
18,864 cells, whatever the training fraction. The runs and logs of earlier sessions come back from
Drive with `--ignore-existing`, so finished stages are skipped and the local copies, never older
than Drive's, are kept. If the restore prints an error, interrupt Run all and fix it first: a
training run without its earlier checkpoint starts over and overwrites the one in Drive.
`DATA` must stay `/content/garimpo-data`: runs record the path of their data and refuse another.

In [ ]:
# @title Mount Drive, unpack the data and restore earlier runs
import os
import time

from google.colab import drive

DRIVE = "/content/drive/MyDrive/garimpo"  # kept between sessions
DATA = "/content/garimpo-data"  # runs record this path, so it never changes
os.environ["GARIMPO_DATA_DIR"] = DATA
SESSION_START = globals().get("SESSION_START", time.time())  # older logs hold no attempt of ours
try:
    drive.mount("/content/drive")
    DRIVE_OK = True
except Exception as error:  # noqa: BLE001 -- a raise here would cancel the release
    print(f"Drive did not mount, so the cells that need it skip: {error}")
    DRIVE_OK = False
!apt-get -qq install -y rsync zstd > /dev/null
!mkdir -p {DATA} runs outputs/logs
if DRIVE_OK:
    !mkdir -p {DRIVE}/runs {DRIVE}/logs
    # The marker is written only after tar succeeds, so a cut unpack is redone, never trusted.
    if Path(DATA, ".unpacked").exists():
        print(f"The data is already in {DATA}.")
    else:
        !tar -I zstd -xf {DRIVE}/data.tar.zst -C {DATA}
        if globals().get("_exit_code") == 0:
            Path(DATA, ".unpacked").touch()
    !rsync -a --ignore-existing --exclude '*.tmp' {DRIVE}/runs/ runs/
    !rsync -a --ignore-existing {DRIVE}/logs/ outputs/logs/
    !df -h /content

## 4. Tests

The test suite runs here on CUDA, where none is skipped, once per commit: a pass is recorded in
`MyDrive/garimpo/tested/`, so a later session on the same commit skips it. When a test fails, no
run starts; the cell says so, and Run all still reaches the release.

In [ ]:
# @title Run the tests on the GPU, once per commit
import subprocess

found = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=False)  # noqa: S607
COMMIT = found.stdout.strip()
TESTED = Path(DRIVE, "tested", COMMIT or "unknown")
try:
    TESTS_PASSED = SETUP_OK and DRIVE_OK and bool(COMMIT) and TESTED.exists()
except OSError as error:  # Drive's mount can drop; testing again is the safe answer
    print(f"Could not read Drive's record of the tests: {error}")
    TESTS_PASSED = False
if not (SETUP_OK and DRIVE_OK):
    print("Skipped: the setup failed or Drive did not mount.")
elif TESTS_PASSED:
    print(f"Skipped: commit {COMMIT[:7]} passed the tests in an earlier session.")
else:
    !GARIMPO_TEST_DEVICE=cuda pytest -q
    TESTS_PASSED = bool(COMMIT) and globals().get("_exit_code") == 0
    if TESTS_PASSED:
        try:
            TESTED.parent.mkdir(parents=True, exist_ok=True)
            TESTED.touch()
        except OSError as error:
            print(f"Could not record the pass in Drive, so the next session tests again: {error}")
    else:
        print("The tests failed, so no run starts: fix the code and push.")

## 5. Runs

The pilot comes first: it trains the four models on 1% of the training tiles for a few updates and
runs every stage on one tile of each evaluation group, to check the code. Until all four pilots pass on the commit checked out, no full
run starts or goes on. Once they pass, in this session or an earlier one, the four full runs start,
or resume from their last checkpoint. `garimpo experiment` runs each model with `garimpo run`, one after another on the
GPU; each prints one line per stage with its time and key numbers, writes everything to
`outputs/logs/<scale>-<model>.log`, and ends with `PIPELINE CHECK: PASS` or `FAIL`. A model that
fails does not stop the next one. A copy of `runs/` and the logs to Drive every 30 minutes keeps the
finished stages if the session ends early; Run all again in a new session resumes from there.

The verdicts, one line per model: `PASS` means every stage finished and every check held. After
`FAIL`, the log names the stage that failed. `NO VERDICT`: the job died before its check.
`NO LOG`: no log yet. `NOT RUN`: this session wrote nothing to that model's log.

Interrupting this cell once stops the current stage, still prints its check, and runs no later
model; then run sections 6 and 7 by hand to save and release.

Then the command merges the four runs' tables of each scale that ran in this session: into
`runs/pilot/tables/` for the pilot, into `outputs/` for the full runs, and this cell copies
`outputs/` to Drive. The merge refuses, and writes nothing, unless every stage of all four runs is
`ok`, the other layers included. After the full runs, download `MyDrive/garimpo/outputs/` and commit
it from your own machine: these are the final tables.

In [ ]:
# @title Run the pilot, then the full runs, then merge each scale that ran
import threading


def copy_to_drive_every_30_min(stop: threading.Event) -> None:
    """Copy runs/ and the logs to Drive every 30 minutes until stop is set."""
    runs = ["rsync", "-a", "--exclude", "*.tmp", "runs/", f"{DRIVE}/runs/"]
    logs = ["rsync", "-a", "outputs/logs/", f"{DRIVE}/logs/"]
    while not stop.wait(1800):
        for command in (runs, logs):
            try:
                copied = subprocess.run(command, check=False)  # noqa: S603
            except OSError as error:
                print(f"The copy to Drive every 30 minutes failed: {error}")
                continue
            if copied.returncode != 0:
                print(f"The copy to Drive every 30 minutes ended with code {copied.returncode}")


periodic_copy = globals().get("periodic_copy")
if DRIVE_OK and (periodic_copy is None or not periodic_copy.is_alive()):
    stop_copy = threading.Event()
    periodic_copy = threading.Thread(
        target=copy_to_drive_every_30_min, args=(stop_copy,), daemon=True
    )
    periodic_copy.start()
if TESTS_PASSED:
    !garimpo experiment {OPTIONS} --since {SESSION_START}
    !rsync -a outputs/ {DRIVE}/outputs/
else:
    print("Skipped: the tests did not pass on this commit.")

## 6. Save

Stops the copy every 30 minutes, after any copy in progress, and copies the runs and logs to Drive
one last time.

In [ ]:
# @title Save to Drive
if periodic_copy is not None:
    stop_copy.set()
    periodic_copy.join()
if DRIVE_OK:
    !rsync -a --exclude '*.tmp' runs/ {DRIVE}/runs/
    !rsync -a outputs/logs/ {DRIVE}/logs/

## 7. Release

Writes the pending files to Drive and releases the GPU. No cell before this one raises a Python
error, so Run all always reaches it.

In [ ]:
# @title Release the runtime
import os

from google.colab import drive, runtime

if RELEASE:
    if os.path.ismount("/content/drive"):
        try:
            drive.flush_and_unmount()
        except Exception as error:  # noqa: BLE001 -- the runtime must be released whatever failed
            print(f"Drive did not confirm the last writes; check the files in Drive: {error}")
    runtime.unassign()
else:
    print("Kept: RELEASE is not checked.")

## Where the files are

| Path | Contents | Kept after the session |
|---|---|---|
| `MyDrive/garimpo/data.tar.zst` | The data, which the notebook only reads | Yes |
| `MyDrive/garimpo/runs/` | Every run: checkpoints, tables, stage statuses | Yes |
| `MyDrive/garimpo/logs/` | The log of every pilot and full-run job | Yes |
| `MyDrive/garimpo/outputs/` | The merged tables of the full runs | Yes |
| `/content/garimpo-uncertainty/` | The code | No |
| `/content/garimpo-data/` | The unpacked data | No |